In [1]:
import pandas as pd
import datasets
from pathlib import Path

c:\Vijay\PyCode\ContentID\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Pull datasets from HuggingFace

In [ ]:
# pull dataset from huggingface
train_dataset = datasets.load_dataset("VijayRam1812/safety_dataset", split="train")
val_dataset = datasets.load_dataset("VijayRam1812/safety_dataset", split="val")
test_dataset = datasets.load_dataset("VijayRam1812/safety_dataset_synthetic_eval", split="train")


## Convert to pandas dataframe

In [ ]:
# convert to pandas dataframe
train_df = train_dataset.to_pandas()
val_df = val_dataset.to_pandas()
test_df = test_dataset.to_pandas()

In [ ]:
train_df.head()

In [ ]:
val_df.head()

## Stratified downsampling

Sample equal counts per (stratify_key, conv_type) cell for balanced conv_type distribution

Train Dataset Downsampling

In [ ]:
# Sample equal counts per (stratify_key, conv_type) cell for balanced conv_type distribution
n_cells = train_df.groupby(["stratify_key", "conv_type"]).ngroups
n_per_cell = max(1, len(train_df) // 2 // n_cells)
min_available = train_df.groupby(["stratify_key", "conv_type"]).size().min()
n_per_cell = min(n_per_cell, min_available)

train_sampled = train_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(
    lambda g: g.sample(n=n_per_cell, random_state=42)
).reset_index(drop=True)

print(f"Original training set: {len(train_df)} samples")
print(f"Sampled training set:  {len(train_sampled)} samples  ({n_per_cell} per cell)")
print(f"\nPer stratify_key counts (before -> after):")
comparison = pd.DataFrame({
    "original": train_df["stratify_key"].value_counts().sort_index(),
    "sampled":  train_sampled["stratify_key"].value_counts().sort_index()
})
comparison["ratio"] = (comparison["sampled"] / comparison["original"]).round(2)
print(comparison.to_string())


Verify the distribution of conv_types in the sampled data

In [ ]:
# find the distribution of conv_types in the sampled data
conv_type_counts = train_sampled["conv_type"].value_counts()
conv_type_percentages = (conv_type_counts / len(train_sampled) * 100).round(2)
print("\nDistribution of conv_types in the sampled training data:")
for conv_type, count in conv_type_counts.items():
    pct = conv_type_percentages[conv_type]
    print(f"{conv_type}: {count} samples ({pct}%)")

Validation Dataset Downsampling

In [ ]:
# Sample equal counts per (stratify_key, conv_type) cell for balanced conv_type distribution
n_cells = val_df.groupby(["stratify_key", "conv_type"]).ngroups
n_per_cell = max(1, len(val_df) // 2 // n_cells)
min_available = val_df.groupby(["stratify_key", "conv_type"]).size().min()
n_per_cell = min(n_per_cell, min_available)

val_sampled = val_df.groupby(["stratify_key", "conv_type"], group_keys=False).apply(
    lambda g: g.sample(n=n_per_cell, random_state=42)
).reset_index(drop=True)

print(f"Original validation set: {len(val_df)} samples")
print(f"Sampled validation set:  {len(val_sampled)} samples  ({n_per_cell} per cell)")
print(f"\nPer stratify_key counts (before -> after):")
comparison = pd.DataFrame({
    "original": val_df["stratify_key"].value_counts().sort_index(),
    "sampled":  val_sampled["stratify_key"].value_counts().sort_index()
})
comparison["ratio"] = (comparison["sampled"] / comparison["original"]).round(2)
print(comparison.to_string())

Distribution of conv_types in the sampled validation data

In [ ]:
# find the distribution of conv_types in the sampled data
conv_type_counts = val_sampled["conv_type"].value_counts()
conv_type_percentages = (conv_type_counts / len(val_sampled) * 100).round(2)
print("\nDistribution of conv_types in the sampled training data:")
for conv_type, count in conv_type_counts.items():
    pct = conv_type_percentages[conv_type]
    print(f"{conv_type}: {count} samples ({pct}%)")

## Save raw data as csv files

In [ ]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "raw"
output_dir.mkdir(parents=True, exist_ok=True)

train_df.to_csv(output_dir / "train_dataset.csv", index=False)
val_df.to_csv(output_dir / "val_dataset.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  train_dataset.csv  : {len(train_df)} rows")
print(f"  val_dataset.csv    : {len(val_df)} rows")

## Save sampled data as csv files

In [ ]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "train"
output_dir.mkdir(parents=True, exist_ok=True)

train_sampled.to_csv(output_dir / "train_sampled.csv", index=False)
val_sampled.to_csv(output_dir / "val_sampled.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  train_sampled.csv  : {len(train_sampled)} rows")
print(f"  val_sampled.csv    : {len(val_sampled)} rows")


In [ ]:
PROJECT_ROOT = Path(r"C:\Vijay\PyCode\ContentID")
output_dir = PROJECT_ROOT / "data" / "test"
output_dir.mkdir(parents=True, exist_ok=True)

test_df.to_csv(output_dir / "test_dataset.csv", index=False)

print(f"Saved to {output_dir}")
print(f"  test_dataset.csv  : {len(test_df)} rows")

## EDA - Exploratory Data Analysis 

Deeper analysis of the data to understand the distribution of the data and tokens and other metrics. 

In [ ]:
# Visualize and print the distribution of the data by labels (benign/harmful) for train, val, and test splits.
import matplotlib.pyplot as plt

def plot_label_distribution(df, split_name):
    label_counts = df['label'].value_counts().sort_index()
    label_names = {0: 'benign', 1: 'harmful'}
    plt.bar(label_counts.index.map(label_names), label_counts.values, color=['skyblue', 'salmon'])
    plt.title(f'Label Distribution in {split_name}')
    plt.ylabel('Count')
    plt.xlabel('Label')
    for i, count in enumerate(label_counts.values):
        plt.text(i, count + 2, str(count), ha='center')
    plt.show()
    print(f"{split_name} label counts:\n{label_counts.rename(index=label_names)}\n")

# You may need to replace `train_df`, `val_df`, `test_df` with the actual DataFrames if their names differ.
plot_label_distribution(train_sampled, "Train")
plot_label_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_label_distribution(test_df, "Test")


In [ ]:
# Understand the distribution of the data by categories

def plot_category_distribution(df, split_name):
    category_counts = df['category'].value_counts().sort_values(ascending=False)
    plt.figure(figsize=(10,5))
    plt.bar(category_counts.index, category_counts.values, color='mediumseagreen')
    plt.title(f'Category Distribution in {split_name}')
    plt.ylabel('Count')
    plt.xlabel('Category')
    for i, count in enumerate(category_counts.values):
        plt.text(i, count + 2, str(count), ha='center', fontsize=9)
    plt.show()
    print(f"{split_name} category counts:\n{category_counts}\n")

plot_category_distribution(train_sampled, "Train")
plot_category_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_category_distribution(test_df, "Test")

In [ ]:
# Visualize and print the distribution of the data by conversation type (user_only, single_turn, multi_turn)
def plot_conv_type_distribution(df, split_name):
    convtype_counts = df['conv_type'].value_counts().sort_index()
    plt.bar(convtype_counts.index, convtype_counts.values, color=['gold', 'lightblue', 'lightgreen'])
    plt.title(f'Conversation Type Distribution in {split_name}')
    plt.ylabel('Count')
    plt.xlabel('Conversation Type')
    for i, count in enumerate(convtype_counts.values):
        plt.text(i, count + 2, str(count), ha='center', fontsize=9)
    plt.show()
    print(f"{split_name} conversation type counts:\n{convtype_counts}\n")

plot_conv_type_distribution(train_sampled, "Train")
plot_conv_type_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_conv_type_distribution(test_df, "Test")

In [ ]:
# Visualize and print the distribution of the data by number of tokens per conversation

from transformers import AutoTokenizer
import numpy as np

# Choose a tokenizer consistent with your model; e.g., "google/gemma-1.1-2b-it"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

def compute_token_lengths(df):
    # Compute tokens of the concatenated raw_text (or join messages if needed)
    # .get if raw_text exists, else reconstruct from messages
    def get_text(row):
        if 'raw_text' in row and pd.notnull(row['raw_text']):
            return row['raw_text']
        elif 'messages' in row and hasattr(row['messages'], '__len__') and not isinstance(row['messages'], str):
            return "\n".join([m.get("content", "") for m in row['messages']])
        else:
            return ""
    texts = df.apply(get_text, axis=1).tolist()
    token_lens = [len(tokenizer.encode(text, add_special_tokens=False)) for text in texts]
    return token_lens

def plot_token_length_distribution(df, split_name):
    token_lens = compute_token_lengths(df)
    plt.figure(figsize=(9,5))
    plt.hist(token_lens, bins=30, color='orchid', alpha=0.75)
    plt.title(f'Token Count Distribution in {split_name}')
    plt.xlabel('Number of Tokens')
    plt.ylabel('Number of Conversations')
    plt.grid(axis='y', alpha=0.2)
    plt.show()
    print(f"{split_name} token count stats: min {min(token_lens)}, max {max(token_lens)}, median {np.median(token_lens)}, mean {np.mean(token_lens):.2f}")
    print()

plot_token_length_distribution(train_sampled, "Train")
plot_token_length_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_token_length_distribution(test_df, "Test")


In [ ]:
import ast

def count_message_objects(msgs):
    """Count JSON objects in the messages field (handles list, numpy array, and string representations)."""
    if hasattr(msgs, '__len__') and not isinstance(msgs, str):
        return len(msgs)
    if isinstance(msgs, str):
        try:
            parsed = ast.literal_eval(msgs)
            if hasattr(parsed, '__len__'):
                return len(parsed)
        except (ValueError, SyntaxError):
            pass
    return 0

def plot_conversation_length_distribution(df, split_name):
    conv_lengths = df['messages'].apply(count_message_objects)
    max_len = max(conv_lengths.max(), 1)
    plt.figure(figsize=(8, 4))
    plt.hist(conv_lengths, bins=range(1, max_len + 2), align='left', color='teal', alpha=0.75, rwidth=0.9)
    plt.title(f'Message Count Distribution in {split_name}')
    plt.xlabel('Number of Message Objects per Conversation')
    plt.ylabel('Number of Samples')
    plt.xticks(range(1, max_len + 1))
    plt.grid(axis='y', alpha=0.2)
    plt.show()
    print(f"{split_name} message count stats: min {conv_lengths.min()}, max {conv_lengths.max()}, median {np.median(conv_lengths)}, mean {np.mean(conv_lengths):.2f}")
    print()

plot_conversation_length_distribution(train_sampled, "Train")
plot_conversation_length_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_conversation_length_distribution(test_df, "Test")

In [ ]:
# Analyze and visualize the distribution of conversation turns (number of messages per conversation) for each data split

def plot_turn_count_distribution(df, split_name):
    turn_counts = df['messages'].apply(count_message_objects)
    plt.figure(figsize=(8, 4))
    plt.hist(turn_counts, bins=range(1, turn_counts.max() + 2), align='left', color='slateblue', alpha=0.7, rwidth=0.9)
    plt.title(f'Number of Turns per Conversation in {split_name}')
    plt.xlabel('Number of Turns')
    plt.ylabel('Number of Conversations')
    plt.xticks(range(1, turn_counts.max() + 1))
    plt.grid(axis='y', alpha=0.2)
    plt.show()
    print(f'{split_name} turn count stats: min={turn_counts.min()}, max={turn_counts.max()}, median={np.median(turn_counts)}, mean={np.mean(turn_counts):.2f}')
    print()

plot_turn_count_distribution(train_sampled, "Train")
plot_turn_count_distribution(val_sampled, "Validation")
if 'test_df' in globals():
    plot_turn_count_distribution(test_df, "Test")

In [2]:
train_sampled = pd.read_csv("../data/train/train_sampled.csv")
val_sampled = pd.read_csv("../data/train/val_sampled.csv")
test_df = pd.read_csv("../data/test/test_dataset.csv")

In [3]:
# Count NaNs in the inputs
nan_counts = pd.DataFrame({
    'Train': train_sampled.isnull().sum(),
    'Validation': val_sampled.isnull().sum(),
    'Test': test_df.isnull().sum()
})
print("NaN counts per column:")
print(nan_counts)
print(f"\nTotal NaNs - Train: {train_sampled.isnull().sum().sum()}, Validation: {val_sampled.isnull().sum().sum()}, Test: {test_df.isnull().sum().sum()}")


NaN counts per column:
              Train  Validation  Test
category          0           0   0.0
conv_type         0           0   0.0
label             0           0   0.0
messages          0           0   0.0
raw_text          0           0   0.0
source            0           0   0.0
stratify_key      0           0   NaN

Total NaNs - Train: 0, Validation: 0, Test: 0


In [5]:
from transformers import AutoTokenizer
import numpy as np

# Choose a tokenizer consistent with your model; e.g., "google/gemma-1.1-2b-it"
tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-it")

def compute_token_lengths(df):
    # Compute tokens of the concatenated raw_text (or join messages if needed)
    # .get if raw_text exists, else reconstruct from messages
    def get_text(row):
        if 'raw_text' in row and pd.notnull(row['raw_text']):
            return row['raw_text']
        elif 'messages' in row and hasattr(row['messages'], '__len__') and not isinstance(row['messages'], str):
            return "\n".join([m.get("content", "") for m in row['messages']])
        else:
            return ""
    texts = df.apply(get_text, axis=1).tolist()
    token_lens = [len(tokenizer.encode(text, add_special_tokens=False)) for text in texts]
    return token_lens

train_counts = compute_token_lengths(train_sampled)
val_counts = compute_token_lengths(val_sampled)
test_counts = compute_token_lengths(test_df)

In [ ]:
print(train_counts)